# SQL, Data Acquisition and Join Validation

In [49]:
import pandas as pd
import sqlite3
import json

customers = pd.DataFrame({
"CustomerID": [
"C001", "C002", "C003", "C004",
"C004", "C005", "C006", "C007",
"C008", "C009", "C010"
],
"CustomerName": [
"Amina", "Brian", "Carol", "David",
"David", "Esther", "Faith", "George",
"Halima", "Ian", "Jane"
],
"Segment": [
"Retail", "Corporate", "Retail", "SME",
"SME", "Corporate", "SME", "Retail",
"Corporate", "Retail", "SME"
],
"RegionCode": [
"NRB", "MSA", "NRB", "KSM",
"KSM", "NKR", "NRB", "MSA",
"KSM", "NKR", "NRB"
]
})
customers

,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C002,Brian,Corporate,MSA
2,C003,Carol,Retail,NRB
3,C004,David,SME,KSM
4,C004,David,SME,KSM
5,C005,Esther,Corporate,NKR
6,C006,Faith,SME,NRB
7,C007,George,Retail,MSA
8,C008,Halima,Corporate,KSM
9,C009,Ian,Retail,NKR


In [50]:
customers.to_csv(
"customers.csv",
index=False
)

In [6]:
import pandas as pd

orders = pd.DataFrame({
    "OrderID": [
        "0001", "0002", "0003", "0004", "0005",
        "0006", "0007", "0008", "0009", "0010",
        "0011", "0012", "0013", "0014", "0015"
    ],
    "CustomerID": [
        "C001", "C002", "C001", "C003", "C004",
        "C005", "C006", "C002", "C007", "C008",
        "C009", "C010", "C006", "C003", "C999"
    ],
    "OrderAmount": [
        4500, 12500, 3200, 6800, 9100,
        15000, 7600, 8300, 2500, 11000,
        5400, 6200, 4300, 7200, 9900
    ],
    "ProductCategory": [
        "Electronics", "Office", "Home", "Electronics", "Home",
        "Office", "Office", "Home", "Electronics", "Electronics",
        "Office", "Home", "Electronics", "Office", "Electronics"
    ]
})

orders.to_csv("orders.csv", index=False)

JSON region lookup

In [9]:
regions = [
    {"RegionCode": "NRB", "RegionName": "Nairobi"},
    {"RegionCode": "MSA", "RegionName": "Mombasa"},
    {"RegionCode": "KSM", "RegionName": "Kisumu"},
    {"RegionCode": "NKR", "RegionName": "Nakuru"}
]
with open("regions.json", "w") as file:
    json.dump(regions, file, indent=4)

inspecting the sources

In [48]:
customers = pd.read_csv("customers.csv")
orders = pd.read_csv("orders.csv")
customers.head()
orders.head()
customers.shape
orders.shape

(15, 4)

reload source CSV files

In [11]:
import pandas as pd
customers = pd.read_csv("customers.csv")
orders = pd.read_csv("orders.csv")

# inspecting dataset

first 5 rows of each dataset

In [12]:
print("--- Customers Head ---")
display(customers.head())

print("--- Orders Head ---")
display(orders.head())

--- Customers Head ---


,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C002,Brian,Corporate,MSA
2,C003,Carol,Retail,NRB
3,C004,David,SME,KSM
4,C004,David,SME,KSM


--- Orders Head ---


,OrderID,CustomerID,OrderAmount,ProductCategory
0,1,C001,4500,Electronics
1,2,C002,12500,Office
2,3,C001,3200,Home
3,4,C003,6800,Electronics
4,5,C004,9100,Home


Q1. What does one row represent in customers.csv?

row one represents:customerID,customer name,segment and region code

Q2. What does one row represent in orders.csv?

one row represents :order id,customer ID,order amount and product category


Q3. Are the two datasets at the same aggregation grain? Explain briefly.

no they not of the same aggregation grain since in customers.csv is one row per customer while orders.csv is 1 row per order where customer can place multiple orders


# Detect key problems

In [15]:
customers["CustomerID"].duplicated().sum()
customers[
customers["CustomerID"].duplicated(keep=False)
]

,CustomerID,CustomerName,Segment,RegionCode
3,C004,David,SME,KSM
4,C004,David,SME,KSM


Q4. Why can a duplicate CustomerID be dangerous when joining customer and order tables?

This causes two significant data integrity problems:   
1:Row Count Inflation: The joined result creates extra duplicate rows for those orders, artificially increasing the total row count of your dataset.  
 2.Metric Distortions: Numerical calculations like SUM(OrderAmount) or revenue KPIs become heavily inflated because sales amounts associated with the duplicated customer key get summed multiple times

# checking unmatched customer IDS

In [16]:
set(orders["CustomerID"]) - set(customers["CustomerID"])

{'C999'}

Q5. Which CustomerID appears in orders but not in customers?

C999 appears in the orders dataset but does not exist in the customers dataset.


Q6. Why should an unmatched record be investigated rather than silently deleted?

Deleting an unmatched transaction row (like C999) removes valid order amounts from your datasets, leading to  incorrect overall sales totals

# Creating the SQLite Database

In [18]:
conn = sqlite3.connect("retail_lab.db")
customers.to_sql(
"customers",
conn,
if_exists="replace",
index=False
)
orders.to_sql(
"orders",
conn,
if_exists="replace",
index=False
)

15

# testing the database

In [19]:
query = """
SELECT *
FROM customers
LIMIT 5;
"""
pd.read_sql_query(query, conn)

,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C002,Brian,Corporate,MSA
2,C003,Carol,Retail,NRB
3,C004,David,SME,KSM
4,C004,David,SME,KSM


# SELECT and WHERE

In [20]:
query_e1 = """
SELECT
    OrderID,
    CustomerID,
    OrderAmount
FROM orders;
"""

df_e1 = pd.read_sql_query(query_e1, conn)
display(df_e1)

,OrderID,CustomerID,OrderAmount
0,1,C001,4500
1,2,C002,12500
2,3,C001,3200
3,4,C003,6800
4,5,C004,9100
5,6,C005,15000
6,7,C006,7600
7,8,C002,8300
8,9,C007,2500
9,10,C008,11000


1. Customers in the Retail segment

In [21]:
query_retail = """
SELECT *
FROM customers
WHERE Segment = 'Retail';
"""

df_retail = pd.read_sql_query(query_retail, conn)
display(df_retail)

,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C003,Carol,Retail,NRB
2,C007,George,Retail,MSA
3,C009,Ian,Retail,NKR


2. Orders with OrderAmount greater than KSh 8,000

In [22]:
query_high_value = """
SELECT *
FROM orders
WHERE OrderAmount > 8000;
"""

df_high_value = pd.read_sql_query(query_high_value, conn)
display(df_high_value)

,OrderID,CustomerID,OrderAmount,ProductCategory
0,2,C002,12500,Office
1,5,C004,9100,Home
2,6,C005,15000,Office
3,8,C002,8300,Home
4,10,C008,11000,Electronics
5,15,C999,9900,Electronics


3. Orders in the Electronics category

In [23]:
query_electronics = """
SELECT *
FROM orders
WHERE ProductCategory = 'Electronics';
"""

df_electronics = pd.read_sql_query(query_electronics, conn)
display(df_electronics)

,OrderID,CustomerID,OrderAmount,ProductCategory
0,1,C001,4500,Electronics
1,4,C003,6800,Electronics
2,9,C007,2500,Electronics
3,10,C008,11000,Electronics
4,13,C006,4300,Electronics
5,15,C999,9900,Electronics


4. Electronics orders worth more than KSh 8,000

In [24]:
query_elec_high_value = """
SELECT *
FROM orders
WHERE ProductCategory = 'Electronics'
  AND OrderAmount > 8000;
"""

df_elec_high_value = pd.read_sql_query(query_elec_high_value, conn)
display(df_elec_high_value)

,OrderID,CustomerID,OrderAmount,ProductCategory
0,10,C008,11000,Electronics
1,15,C999,9900,Electronics


# GROUP BY and Aggregate Fuctions

In [27]:
query_f = """
SELECT
    ProductCategory,
    COUNT(*) AS NumberOfOrders,
    SUM(OrderAmount) AS TotalSales,
    AVG(OrderAmount) AS AverageOrderValue
FROM orders
GROUP BY ProductCategory;
"""

df_f = pd.read_sql_query(query_f, conn)
display(df_f)

,ProductCategory,NumberOfOrders,TotalSales,AverageOrderValue
0,Electronics,6,39000,6500.0
1,Home,4,26800,6700.0
2,Office,5,47700,9540.0


Q7. Write two Short Observations About Category Performance

1.Electronics drives overall volume and top-line sales

2.Home has the highest Average Order Value


# JOIN Customers and Orders

In [47]:
query_g = """
SELECT
    o.OrderID,
    o.CustomerID,
    c.CustomerName,
    c.Segment,
    o.OrderAmount
FROM orders o
LEFT JOIN customers c
    ON o.CustomerID = c.CustomerID;
"""

joined = pd.read_sql_query(query_g, conn)
display(joined)

print("Original orders count:", len(orders))
print("Joined dataframe count:", len(joined))

,OrderID,CustomerID,CustomerName,Segment,OrderAmount
0,1,C001,Amina,Retail,4500
1,2,C002,Brian,Corporate,12500
2,3,C001,Amina,Retail,3200
3,4,C003,Carol,Retail,6800
4,5,C004,David,SME,9100
5,5,C004,David,SME,9100
6,6,C005,Esther,Corporate,15000
7,7,C006,Faith,SME,7600
8,8,C002,Brian,Corporate,8300
9,9,C007,George,Retail,2500


Original orders count: 15
Joined dataframe count: 16


Q8. Are the row counts equal? If not, explain exactly what caused the difference.

No, the row counts are not equal.
len(orders) is 15.   len(joined) is 16.

  Cause: The customer table contains a duplicate key for CustomerID C004 (David). When order 0005 (placed by C004) was joined, SQL matched it twice—once against each duplicate row for C004 in the customers table—creating 1 extra row in the resulting joined dataset.

Q9. Locate the unmatched customer record in the joined output. What values appear for CustomerName and Segment?

The unmatched record is order 0015 for CustomerID C999.
C999 does not exist in customers.csv, the LEFT JOIN produces None (NaN / NULL) for both CustomerName and Segment.

# Reconcile the totals

# Before the Join



In [30]:
orders["OrderAmount"].sum()

np.int64(113500)

After the join

In [31]:
joined["OrderAmount"].sum()

np.int64(122600)

In [33]:
pre_join_total = orders["OrderAmount"].sum()
post_join_total = joined["OrderAmount"].sum()

print("Total Sales BEFORE JOIN: KSh", pre_join_total)
print("Total Sales AFTER JOIN:  KSh", post_join_total)
print("Difference:              KSh", post_join_total - pre_join_total)

Total Sales BEFORE JOIN: KSh 113500
Total Sales AFTER JOIN:  KSh 122600
Difference:              KSh 9100


Q10. Are the two sales totals equal?

No



Q11. If the total changed, which customer caused the duplication?

Customer C004

Q12. Why is total reconciliation an important validation step after a JOIN?

It confirms whether records were accidentally dropped or omitted during transformation steps.

# Corrected the Duplicate-Key Problem

In [36]:

customers_clean = (
    customers
    .drop_duplicates(subset="CustomerID")
)
customers_clean.to_sql(
    "customers_clean",
    conn,
    if_exists="replace",
    index=False
)
query = """
SELECT
    o.OrderID,
    o.CustomerID,
    c.CustomerName,
    c.Segment,
    o.OrderAmount
FROM orders o
LEFT JOIN customers_clean c
    ON o.CustomerID = c.CustomerID;
"""

clean_joined = pd.read_sql_query(query, conn)
clean_joined
len(orders), len(clean_joined)
orders["OrderAmount"].sum(), clean_joined["OrderAmount"].sum()

(np.int64(113500), np.int64(113500))

Q13. Has the corrected JOIN preserved both the row count and the total sales value? Explain.

Yes it has

# Customer Segment KPIs

In [37]:
query_j = """
SELECT
    COALESCE(c.Segment, 'Unmatched / NULL') AS Segment,
    COUNT(o.OrderID) AS Orders,
    SUM(o.OrderAmount) AS TotalSales,
    AVG(o.OrderAmount) AS AverageOrderValue
FROM orders o
LEFT JOIN customers_clean c ON o.CustomerID = c.CustomerID
GROUP BY COALESCE(c.Segment, 'Unmatched / NULL');
"""

df_segment_kpis = pd.read_sql_query(query_j, conn)
display(df_segment_kpis)

,Segment,Orders,TotalSales,AverageOrderValue
0,Corporate,4,46800,11700.000000
1,Retail,6,29600,4933.333333
2,SME,4,27200,6800.000000
3,Unmatched / NULL,1,9900,9900.000000


Corporate customers spend the most per order since their avarage was the highest.SME brought the most revenue overall and theres missing cutomer data to fix.one order isnt linked to any customer name or segment.

# merging JSON Region lookup

In [42]:
regions_df = pd.read_json("regions.json")
regions_df


,RegionCode,RegionName
0,NRB,Nairobi
1,MSA,Mombasa
2,KSM,Kisumu
3,NKR,Nakuru


In [43]:
customers_regions = customers_clean.merge(
    regions_df,
    on="RegionCode",
    how="left"
)
display(customers_regions.head())

,CustomerID,CustomerName,Segment,RegionCode,RegionName
0,C001,Amina,Retail,NRB,Nairobi
1,C002,Brian,Corporate,MSA,Mombasa
2,C003,Carol,Retail,NRB,Nairobi
3,C004,David,SME,KSM,Kisumu
4,C005,Esther,Corporate,NKR,Nakuru


Q15. Confirm that each customer now has a readable RegionName rather than only a RegionCode.

Yes they have


Q16. If a RegionCode did not exist in regions.json, what would a left merge produce for RegionName?

segment

# Regional Sales Analysis

In [45]:
customers_regions.to_sql("customers_regions", conn, if_exists="replace", index=False)
query_regional = """
SELECT
    COALESCE(cr.RegionName, 'Unmatched / NULL') AS Region,
    COUNT(o.OrderID) AS NumberOfOrders,
    SUM(o.OrderAmount) AS TotalSales
FROM orders o
LEFT JOIN customers_regions cr ON o.CustomerID = cr.CustomerID
GROUP BY COALESCE(cr.RegionName, 'Unmatched / NULL');
"""

df_regional_kpis = pd.read_sql_query(query_regional, conn)
display(df_regional_kpis)

,Region,NumberOfOrders,TotalSales
0,Kisumu,2,20100
1,Mombasa,3,23300
2,Nairobi,7,39800
3,Nakuru,2,20400
4,Unmatched / NULL,1,9900


Q17. Which region generated the highest sales?

Nairobi

Q18. Should management automatically conclude that this is the “best-performing” region? Explain using at
least two limitations of the available data.

No
it has the most orders but the income ratio is low compared to others


# CONCLUSION

The biggest data-quality problem was a duplicate customer ID (C004) in the customer table. This caused extra rows to be created during the JOIN, which double-counted orders and made our sales look higher than they actually were. Checking sales totals after joining tables is important to make sure no data was duplicated or lost. Looking at the cleaned data, the SME customer segment generated the most total sales. One limitation of this analysis is that we only looked at total sales and did not have any profit or expense data.